In [1]:
import torch
from src.mssp import MSSP
import matplotlib.pyplot as plt
import pandas as pd
import time

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
RANDOM_SEED = 42

In [2]:
data = pd.read_csv("bikes_data.csv")

In [3]:
data.head()

,Temp,Atemp,Hum,Windspeed,Casual,Registered,Cntlag,Cnt
0,0.200000,0.212122,0.590436,0.160296,120.0,1229.0,5.832896,5.458242
1,0.226957,0.229270,0.436958,0.186900,108.0,1454.0,5.458242,5.335675
2,0.204348,0.233209,0.518262,0.089565,82.0,1518.0,5.335675,5.315381
3,0.196522,0.208839,0.498697,0.168726,88.0,1518.0,5.315381,5.250001
4,0.165000,0.162254,0.535834,0.266804,148.0,1362.0,5.250001,4.857664


In [4]:
X = torch.tensor(data.drop(columns=['Cnt']).values, dtype=torch.double)
y = torch.tensor(data['Cnt'].values, dtype=torch.double)

In [5]:
n = len(X)
train_end = int(n * 0.7)
valid_end = int(n * 0.8)

X_train = X[:train_end]
X_valid = X[train_end:valid_end]
X_test = X[valid_end:]

y_train = y[:train_end]
y_valid = y[train_end:valid_end]
y_test = y[valid_end:]

In [6]:
mae_scores = []
for i in range(1, 11):
    mssp = MSSP(
        n_best=100, 
        loss_fn="mse", 
        random_seed=42, 
        epochs=i, 
        diversity_ratio=0.75, 
        pow_cross=True,
    )

    st_ft = time.time()
    mssp.fit(X_train, y_train, X_valid, y_valid)
    et_fr = time.time()
    for k in (1, 4, 16, 32):
        st_pr = time.time()
        mssp_pred = mssp.predict(X_test, top_k=k)
        et_pr = time.time()
        # Remove nans and infs for valid mask
        mask = (~mssp_pred.isnan()) & (~torch.isinf(mssp_pred))
        mae = mean_absolute_error(mssp_pred[mask], y_test[mask])
        mae_scores.append(('mssp', i, k, mae, et_fr - st_ft, et_pr - st_pr))

self.i tensor([ 0,  0,  0,  ..., 53, 53, 54]) torch.Size([1540])
self.j tensor([ 1,  2,  3,  ..., 54, 55, 55]) torch.Size([1540])
self.coef_ tensor([[ -0.5702,   1.0771],
        [ -0.3668,   0.0386],
        [  1.6670,   0.8193],
        ...,
        [ 26.6369, -22.7123],
        [-30.7639,  25.4676],
        [-12.8307,  12.4198]], device='cuda:0') torch.Size([1540, 2])
self.intercept_ tensor([  2.6205,   7.0595,  -7.8890,  ..., -15.0398,  32.7362,   7.4272],
       device='cuda:0') torch.Size([1540])
Xb torch.Size([73, 56])
self.i tensor([ 0,  0,  0,  ..., 53, 53, 54]) torch.Size([1540])
self.j tensor([ 1,  2,  3,  ..., 54, 55, 55]) torch.Size([1540])
self.coef_ tensor([[ -0.5702,   1.0771],
        [ -0.3668,   0.0386],
        [  1.6670,   0.8193],
        ...,
        [ 26.6369, -22.7123],
        [-30.7639,  25.4676],
        [-12.8307,  12.4198]], device='cuda:0') torch.Size([1540, 2])
self.intercept_ tensor([  2.6205,   7.0595,  -7.8890,  ..., -15.0398,  32.7362,   7.4272],
   

KeyboardInterrupt: 

In [ ]:
linreg_model = LinearRegression()
st_ft = time.time()
linreg_model.fit(X_train, y_train)
et_ft = time.time()
st_pr = time.time()
linreg_pred = torch.tensor(linreg_model.predict(X_test), dtype=torch.double)
et_pr = time.time()
mae = mean_absolute_error(linreg_pred, y_test)
mae_scores.append(("lin", 1, 1, mae, et_ft - st_ft, et_pr - st_pr))

In [ ]:
data['Cntlag'].values

In [ ]:
df = pd.DataFrame(mae_scores)
df.columns = ['model', 'epochs', 'n_models', 'mae', 'fit_time', 'pred_time']
df = df.sort_values('mae')
df = df.reset_index(drop=True)

In [ ]:
df

In [ ]:
k1 = df[(df['n_models'] == 1) & (df['model'] == 'mssp')].sort_values('epochs')
k4 = df[df['n_models'] == 4].sort_values('epochs')
k16 = df[df['n_models'] == 16].sort_values('epochs')
k32 = df[df['n_models'] == 32].sort_values('epochs')

# x_labels = list(range(1, 11))
# x_range = list(range(10))

# plt.plot(x_range, k1['mae'].values[:11], color='red', label='1')
# plt.plot(x_range, k4['mae'].values[:11], color='blue', label='4')
# plt.plot(x_range, k16['mae'].values[:11], color='green', label='16')
# plt.plot(x_range, k32['mae'].values[:11], color='orange', label='32')
# plt.xticks(x_range, x_labels)
plt.plot(k1['mae'].values, color='red', label='1')
plt.plot(k4['mae'].values, color='blue', label='4')
plt.plot(k16['mae'].values, color='green', label='16')
plt.plot(k32['mae'].values, color='orange', label='32')
plt.legend()

In [ ]:
# plt.plot(y_test, color='red')
# plt.plot(linreg_pred)

In [ ]:
# rforest_model = RandomForestRegressor()
# rforest_model.fit(X_train, y_train)

In [ ]:
# rforest_pred = torch.tensor(rforest_model.predict(X_test), dtype=torch.double)
# # plt.plot(inverse_transform_spy(rforest_pred, close_test)[-50:])
# # plt.plot(inverse_transform_spy(y_test, close_test)[-51:-1])
# mean_squared_error(rforest_pred, y_test)

In [ ]:
# svr_model = SVR()
# svr_model.fit(X_train, y_train)
# svr_pred = torch.tensor(svr_model.predict(X_test), dtype=torch.double)
# # plt.plot(inverse_transform_spy(svr_pred, close_test)[-50:])
# # plt.plot(inverse_transform_spy(y_test, close_test)[-51:-1])
# mean_squared_error(svr_pred, y_test)

In [ ]:
mssp.history

In [ ]:
# for i in range(9):
    # print(len([n for n in mssp.model[0].nodes.keys() if isinstance(n, tuple) and n[0] == i]))
[n for n in mssp.model[0].nodes.items() if isinstance(n[0], tuple) and n[0][0] == 8]

In [ ]:
mssp.plot_graph(1)